# 03 Data Wrangling — Automated（整合版）

**Mirrors the CONFIG/engine structure of `01_data_exploration_auto.py` / `02_data_pattern_analysis_auto.ipynb`.**

**目的**（跟 R 版本 `03_data_wrangling.qmd` 的 Executive Summary 一致）：把 D1/D2/D3 的 artist/song 欄位標準化（清掉合作/版本標記、統一大小寫、統一欄位命名），為 04 合併資料做準備，並產生一組 `join_key`（`artist_clean|song_clean`）讓 04 可以直接拿去比對三個資料集。

- **CONFIG（要改的東西）**：`CLEANING_RULES`（直接沿用 02 Step 6 的最終交付物，不重新分析）+ 欄位命名對照表
- **Engine（不用改）**：`clean_column()` 吃一份 `CLEANING_RULES` 條目就能清任何一欄，不分 artist 還是 song
- **輸出**：清理前後健康報告（改了多少、改了什麼）→ 統一命名後的 `artist_clean`/`song_clean` → `join_key` → 存檔給 04 用

## 整體架構：四塊怎麼互相呼叫（先看大局，再看 Step 細節）

跟 02 的結構很像，但 03 沒有「機器算、人判斷」這件事——**該判斷的事情 02 已經做完了**，`CLEANING_RULES` 就是 02 人工判斷完的結果，03 純粹是「照規則執行」，所以 03 只有三塊，不是四塊：

| ① CONFIG | | ② Engine（1 個通用 function） | | ③ Step 1-6（呼叫②，印報表+存檔） |
|---|:---:|---|:---:|---|
| `CLEANING_RULES`（沿用 02）<br>`DATASETS` 欄位/命名對照 | → | `clean_column()`<br>吃一份 rules 條目，不管是 artist 還是 song 規則都能處理，因為 02 已經把兩種規則統一成同一個 schema | → | Step 1：欄位命名對齊<br>Step 2：套用清理規則<br>Step 3：健康報告（清理前後對照）<br>Step 4：建立 Join Key<br>Step 5：跨資料集 Key 重疊率<br>Step 6：存檔 |

**對應到「白板模型」**：
- **① CONFIG** 這次不是重新分析出來的，是**直接把 02 的 Step 6 輸出貼過來**——這就是為什麼 02 最後要堅持統一 schema 的原因：如果 D3 還是用 `"action": "trim_whitespace_only"` 這種不同格式，`clean_column()` 就沒辦法用同一段邏輯處理所有資料集
- **② Engine** 只有一個函數，因為 02 已經把「artist 規則」跟「song 規則」都設計成同一種 key 集合（`remove_after_marker`/`remove_parens_if_contains`/`remove_version_tags_in_parens`/`remove_version_tags_after_dash`/`remove_brackets_entirely`），`clean_column()` 只要檢查這份規則字典裡有哪些 key，就知道要做哪些動作，不用另外寫 `clean_artist()`跟`clean_song()`兩個函數（這是跟 R 版本最大的不同——R 每個資料集手寫兩個清理函數，Python 一個 `clean_column()` 通吃六種欄位）
- **③ Step 1-6** 才是真正把三個資料集的資料送進①②去跑，每跑完一步印一份報表

**這就是為什麼 03 沒有「誰下結論」這一欄**：02 已經把所有需要人工判斷的事情做完了，03 全部都是「機器」——如果 03 執行完發現某個清理結果看起來不合理，那代表要回頭修 02 的 `CLEANING_RULES`，不是在 03 這裡另外加人工判斷的邏輯。

## 03 在幹嘛？—— 6 步驟藍圖

| Step | 做什麼（對應 function） | 問的問題 |
|------|------|------|
| **Step 1** | 欄位命名對齊 | D2 的 `track`、D3 的 `artist_name`/`track_name` 要不要改叫跟 D1 一樣的 `artist`/`song`？（要，04 合併需要統一欄名） |
| **Step 2** | 套用清理規則（`clean_column`） | 把 02 訂好的 `CLEANING_RULES` 實際套到每一欄，清掉該清的合作/版本標記 |
| **Step 3** | 健康報告（清理前後對照） | 到底改了多少筆？改了什麼？清理結果看起來合理嗎？ |
| **Step 4** | 建立 Join Key | 用清理後的 `artist_clean` + `song_clean` 兜出一組能拿去比對三個資料集的鑰匙 |
| **Step 5** | 跨資料集 Key 重疊率預測 | D1 vs D2、D1 vs D3 大概有多少比例對得上？04 合併前先有心理準備 |
| **Step 6** | 存檔 | 清理完的資料存成 `.pkl`，讓 04 能接著讀 |

> **範圍提醒**：R 版本每個資料集各手寫兩個清理函數（`clean_d1_artist`/`clean_d1_song`…共 6 個），邏輯內容照實移植進 `CLEANING_RULES`（02 已經做過這一步），這裡沒有新增或減少清理規則本身，只是換一種「資料驅動」的方式執行同一批規則。

In [1]:
import pandas as pd
import re

# Load validated datasets from Stage 01
d1_billboard   = pd.read_pickle('../Data/cleaned_D1.pkl')
d2_spotify_all = pd.read_pickle('../Data/cleaned_D2.pkl')
d3_music       = pd.read_pickle('../Data/cleaned_D3.pkl')

print('Datasets loaded')
print(f'D1 (Billboard): {len(d1_billboard):,} rows')
print(f'D2 (Spotify):   {len(d2_spotify_all):,} rows')
print(f'D3 (Music):     {len(d3_music):,} rows')

Datasets loaded
D1 (Billboard): 330,087 rows
D2 (Spotify):   41,106 rows
D3 (Music):     28,372 rows


---
## ① CONFIG — `CLEANING_RULES`（直接沿用 02 Step 6）+ 欄位對照表

**`CLEANING_RULES` 這格是直接把 02 notebook 最後 Step 6 印出來的內容複製過來的，不是重新分析**——02 的工作到「決定該清什麼」為止，03 接手「真的把它清掉」。

**欄位對照表（`DATASETS`）多了兩個 02 沒有的欄位**：
- `lowercase`：要不要轉小寫。**2026/07/20 查證更新**：R 的 `clean_d1_artist`/`clean_d2_artist` 都有 `str_to_lower()`，`clean_d3_artist` 卻沒有——原本懷疑是 R 寫漏了，會影響 join 比對率。**實際查證 D3 原始資料**（`artist_name`/`track_name`，28,372 筆）**發現 0 筆含有任何大寫字母，資料來源本身就已經全部是小寫**。代表 R 沒轉小寫不是疏漏造成的資料品質問題——轉不轉結果一樣，不影響 join。**這裡把 D3 也設成 `lowercase: True`，不是在修 bug，是防禦性寫法**：現在零成本零風險（已經是小寫，轉了也不會變），但能防範未來換一批新版 CSV、混進大寫資料時沒被涵蓋到
- `save_path`：Step 6 存檔用

In [2]:
# ============================================================
# CONFIG -- Only edit this section when adding datasets/rules
# ============================================================

# Copied verbatim from 02_data_pattern_analysis_auto.ipynb Step 6 output (2026/07/20 version).
# Artist-type columns:  remove_after_marker / remove_parens_if_contains / preserve / note
# Song-type columns:    remove_version_tags_in_parens / remove_version_tags_after_dash /
#                        remove_brackets_entirely / preserve / note
CLEANING_RULES = {
    "D1": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with\b"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", ",", "brackets not matching collab markers"],
            "note": "41% of '&' and 52% of ',' cases have a collab marker -- real cleanup needed, not just cosmetic",
        },
        "song": {
            # 2026/07/27 corrected against R's actual clean_d1_song() regex (03_data_wrangling.qmd
            # lines 75-89) -- the previous version here was 02's generalized guess, not a literal
            # match to R's code, and it over-stripped (e.g. bare "live" after a dash, which R never
            # does for D1) and under-stripped ("acoustic"/"unplugged"/"featuring"/"ft." in parens,
            # which R does remove for D1). See references/pipeline-history.md for the full diagnosis.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\."],
            "remove_version_tags_after_dash": [r"remix"],  # R ONLY strips remix after a dash for D1, nothing else
            "remove_brackets_entirely": True,
            "preserve": ["standalone parens/dash treated as title subtitle",
                         "'live'/'part'/'take' as lyric words -- false positive risk, do not blanket-remove"],
            "note": "Corrected 2026/07/27 to match R's clean_d1_song() exactly (see note above)",
        },
    },
    "D2": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with\b"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", "/", "+", "x", "commas"],
            "note": "2026/07/20 evidence: 41% of '&' and 52% of ',' cases have a collab marker, "
                    "59% of parens are collab-related -- essentially the same profile as D1, "
                    "previously missing remove_parens_if_contains (D2 parens were wrongly left as blanket-preserve)",
        },
        "track": {
            # 2026/07/27 corrected against R's actual clean_d2_track() regex (03_data_wrangling.qmd
            # lines 163-182). Two real gaps found: (1) R only removes "(live version|acoustic|unplugged)"
            # from parens, and only removes a dash clause that is EXACTLY "live"/"version live"/"live
            # version" (anchored to end of string) -- the old bare "live" tag here matched ANY parens or
            # dash content containing "live" (e.g. "(Live)", "- Live @ Wacken", "- Live / Take 1"), which
            # R does not touch. (2) R also strips "(featuring|feat.|ft.)" from parens for D2 -- the old
            # config only had "feat\." so "(Featuring X)" was never caught. See references/pipeline-history.md.
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live version", r"acoustic",
                                               r"unplugged", r"featuring", r"feat\.", r"ft\.",
                                               r"radio edit"],
            "remove_version_tags_after_dash": [r"remaster", r"remix", r"radio edit", r"feat\."],
            # bare "live" deliberately dropped -- R only removes it as an EXACT dash-clause match
            # ("- Live" / "- Live Version" and nothing else), which this engine's loose "contains"
            # matching cannot safely replicate without over-matching things like "- Live @ Wacken".
            "remove_brackets_entirely": True,
            "preserve": ["'Part'/'Pt.' after dash -- track numbering, not a version tag"],
            "note": "Corrected 2026/07/27 to match R's clean_d2_track() exactly (see note above)",
        },
    },
    "D3": {
        "artist_name": {
            "remove_after_marker": [],
            "remove_parens_if_contains": [],
            "preserve": ["&", ",", "essentially clean already"],
            "note": "2026/07/20 evidence: 0% Featuring, 100% of '&' cases are Simple (no collab marker), "
                    "only 1 row has parentheses at all (Group Info, preserve) -- no removal rules needed, "
                    "still gets the universal trim/normalize step in 03 like every other column",
        },
        "track_name": {
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"feat\.",
                                               r"live version", r"acoustic", r"unplugged",
                                               r"radio edit", r"album version", r"single version"],
            "remove_version_tags_after_dash": [],
            "remove_brackets_entirely": True,
            "preserve": ["'live' outside parens is frequently a lyric word ('as long as i live') -- "
                          "only the precise phrases above (e.g. 'live version', not bare 'live') get removed"],
            "note": "2026/07/20 evidence: Featuring 23.8%, Performance Version 1.77%, Release Version Tag 0.50% "
                    "all showed up in real classification -- added those two categories that were previously "
                    "missing from the removal list. 0% dash usage, so remove_version_tags_after_dash is empty.",
        },
    },
}

# Column mapping + per-dataset execution settings -- the ONLY thing that
# actually differs between D1 / D2 / D3
DATASETS = {
    "D1": {
        "df": d1_billboard,
        "artist_col": "artist", "song_col": "song",
        "lowercase": True,
        "save_path": r"..\Data\wrangled_D1.pkl",
    },
    "D2": {
        "df": d2_spotify_all,
        "artist_col": "artist", "song_col": "track",
        "lowercase": True,
        "save_path": r"..\Data\wrangled_D2.pkl",
    },
    "D3": {
        "df": d3_music,
        "artist_col": "artist_name", "song_col": "track_name",
        "lowercase": True,  # 2026/07/20 查證：D3 原始資料 0 筆含大寫字母，本來就是小寫；
                            # 這裡設 True 是防禦性寫法（零成本零風險），不是修正真的存在的 bug
        "save_path": r"..\Data\wrangled_D3.pkl",
    },
}

---
## ② Engine — No edits needed below this line

`clean_column()` 是整份 03 唯一的清理邏輯，通吃 6 種欄位（D1/D2/D3 各自的 artist + song）。它不知道自己在清哪個資料集，只知道「給我一份文字欄位 + 一份規則字典，我照著做」——這正是 CONFIG/Engine 分離原則的示範：R 版本要手寫 6 個清理函數，這裡只要 1 個。

執行順序（跟 R 每個 `clean_d*_*` 函數內部的順序一致）：
1. 轉小寫（依 `lowercase` 參數決定）+ 去頭尾空白
2. 整個中括號 `[...]` 移除（`remove_brackets_entirely`）
3. 括號內容符合關鍵字就整組括號移除（`remove_parens_if_contains` / `remove_version_tags_in_parens`，兩個 key 都檢查，因為 artist 規則跟 song 規則用的 key 名稱不同）
4. 合作標記後面全部砍掉（`remove_after_marker`）
5. 破折號後內容符合關鍵字就從破折號開始砍掉（`remove_version_tags_after_dash`）
6. 再去一次頭尾空白（前面每個步驟都可能留下多餘空格）

`cleaning_health_report()` 是 Step 3 用的——不是判斷該不該清（那是 02 做的事），是單純**驗證清理結果**：改了幾筆、改了什麼，讓人能一眼看出清理有沒有跑歪。

In [3]:
def clean_column(series, rules, lowercase=True):
    """Generic column cleaner -- reads ONE CLEANING_RULES entry (from 02's Step 6)
    and applies whichever removal keys are present. Works identically for
    artist-type rules (remove_after_marker / remove_parens_if_contains) and
    song-type rules (remove_version_tags_in_parens / remove_version_tags_after_dash /
    remove_brackets_entirely) -- it just checks which keys exist, does NOT decide
    what should be cleaned (that judgment already happened in 02)."""

    cleaned = series.astype(str)
    if lowercase:
        cleaned = cleaned.str.lower()
    cleaned = cleaned.str.strip()

    # Step a: whole [...] bracket removal
    if rules.get("remove_brackets_entirely"):
        cleaned = cleaned.str.replace(r'\s*\[.*?\]', '', regex=True)

    # Step b: parenthetical content -- remove the WHOLE (...) group if its
    # content matches any of these keywords. Two possible key names because
    # artist rules and song rules ask slightly different questions (02's design).
    paren_tags = rules.get("remove_parens_if_contains", []) + rules.get("remove_version_tags_in_parens", [])
    for tag in paren_tags:
        cleaned = cleaned.str.replace(rf'\s*\([^)]*{tag}[^)]*\)', '', regex=True, case=False)

    # Step c: bare collab marker (not in parentheses) -- strip everything after it
    for marker in rules.get("remove_after_marker", []):
        cleaned = cleaned.str.replace(rf'\s+{marker}.*$', '', regex=True, case=False)

    # Step d: content after a ' - ' dash -- strip from the dash onward if it matches
    # a version tag. Uses the SAME ' - ' (space-hyphen-space) delimiter as 02's
    # detection regex, so what gets removed here matches what Step 4 classified.
    for tag in rules.get("remove_version_tags_after_dash", []):
        cleaned = cleaned.str.replace(rf' - .*{tag}.*$', '', regex=True, case=False)

    cleaned = cleaned.str.strip()
    return cleaned


def cleaning_health_report(df, orig_col, clean_col, n_examples=5):
    """Compare original vs cleaned column -- how much changed, and what did it
    look like. Not a judgment call, just a factual before/after check."""
    changed_mask = df[orig_col] != df[clean_col]
    n_changed = int(changed_mask.sum())
    pct_changed = round(n_changed / len(df) * 100, 2)
    print(f"  Changed: {n_changed:,} rows ({pct_changed}%)")

    examples = df[changed_mask][[orig_col, clean_col]].drop_duplicates().head(n_examples)
    for _, row in examples.iterrows():
        print(f"    '{row[orig_col]}'  ->  '{row[clean_col]}'")

    return n_changed, pct_changed

---
## ③ Step 1 — 欄位命名對齊（對應藍圖表 Step 1）

D2 的 `track` 改叫 `song`，D3 的 `artist_name`/`track_name` 改叫 `artist`/`song`，跟 D1 統一——04 合併資料時才能用同一個欄名互相對照。改完之後，`DATASETS` 裡的 `artist_col`/`song_col` 也跟著更新成統一後的名字，後面 Step 2-6 全部只認 `artist`/`song`，不用再記三套不同欄名。

In [4]:
print("[STEP 1] Column Name Alignment（欄位命名對齊）")

RENAME_MAP = {
    "D1": {},                                          # already artist/song
    "D2": {"track": "song"},                            # artist stays artist
    "D3": {"artist_name": "artist", "track_name": "song"},
}

for dataset_name, cfg in DATASETS.items():
    rename_map = RENAME_MAP[dataset_name]
    if rename_map:
        cfg["df"] = cfg["df"].rename(columns=rename_map)
        cfg["artist_col"] = rename_map.get(cfg["artist_col"], cfg["artist_col"])
        cfg["song_col"]   = rename_map.get(cfg["song_col"], cfg["song_col"])
        print(f"  {dataset_name}: renamed {list(rename_map.keys())} -> {list(rename_map.values())}")
    else:
        print(f"  {dataset_name}: no rename needed (already artist/song)")

print("\nColumns now standardized:")
for dataset_name, cfg in DATASETS.items():
    print(f"  {dataset_name}: artist_col='{cfg['artist_col']}', song_col='{cfg['song_col']}'")

[STEP 1] Column Name Alignment（欄位命名對齊）
  D1: no rename needed (already artist/song)
  D2: renamed ['track'] -> ['song']
  D3: renamed ['artist_name', 'track_name'] -> ['artist', 'song']

Columns now standardized:
  D1: artist_col='artist', song_col='song'
  D2: artist_col='artist', song_col='song'
  D3: artist_col='artist', song_col='song'


---
## ③ Step 2 — 套用清理規則（對應藍圖表 Step 2）

呼叫 `clean_column()`，把 `CLEANING_RULES` 真正套到每個資料集的 artist/song 欄位上，產生 `artist_clean`/`song_clean` 兩個新欄位（原始欄位保留，方便 Step 3 對照）。

`CLEANING_RULES` 的 key 是照原始欄位名稱寫的（D2 是 `"track"`、D3 是 `"artist_name"`/`"track_name"`），Step 1 改了欄名之後，這裡用 `RULES_KEY_MAP` 把「現在的欄名」對回「CLEANING_RULES 裡的原始 key」，避免要去改 02 交出來的 CONFIG 本身。

In [5]:
print("[STEP 2] Apply Cleaning Rules（套用清理規則）")

# Maps "current column name" -> "CLEANING_RULES key" per dataset (see Step 1's rename)
RULES_KEY_MAP = {
    "D1": {"artist": "artist", "song": "song"},
    "D2": {"artist": "artist", "song": "track"},
    "D3": {"artist": "artist_name", "song": "track_name"},
}

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]
    lowercase = cfg["lowercase"]
    rules_keys = RULES_KEY_MAP[dataset_name]

    artist_rules = CLEANING_RULES[dataset_name][rules_keys["artist"]]
    song_rules   = CLEANING_RULES[dataset_name][rules_keys["song"]]

    df["artist_clean"] = clean_column(df[artist_col], artist_rules, lowercase=lowercase)
    df["song_clean"]   = clean_column(df[song_col], song_rules, lowercase=lowercase)

    print(f"  {dataset_name}: artist_clean + song_clean created (lowercase={lowercase})")

[STEP 2] Apply Cleaning Rules（套用清理規則）


  D1: artist_clean + song_clean created (lowercase=True)


  D2: artist_clean + song_clean created (lowercase=True)


  D3: artist_clean + song_clean created (lowercase=True)


---
## ③ Step 3 — 健康報告：清理前後對照（對應藍圖表 Step 3）

**這是 R 版本沒有系統性做的事**——R 每個資料集各自貼了一段 Validation 程式碼看前 20 筆改變，這裡用 `cleaning_health_report()` 統一跑六次，順便留意 D3 的 `lowercase=False` 這個決定看起來合不合理（下面會特別註記）。

In [6]:
print("[STEP 3] Cleaning Health Report（清理前後對照）")

health_report = {}
for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name}")
    print(f"{'='*55}")

    print(f"\n=== {dataset_name} artist: '{artist_col}' -> 'artist_clean' ===")
    artist_changed, artist_pct = cleaning_health_report(df, artist_col, "artist_clean")

    print(f"\n=== {dataset_name} song: '{song_col}' -> 'song_clean' ===")
    song_changed, song_pct = cleaning_health_report(df, song_col, "song_clean")

    health_report[dataset_name] = {
        "artist_changed": artist_changed, "artist_pct": artist_pct,
        "song_changed": song_changed, "song_pct": song_pct,
    }

print(f"\n{'='*55}")
print("  NOTE on D3 lowercase (2026/07/20 verified)")
print(f"{'='*55}")
print("  R's clean_d3_artist() never lowercased -- looked like a possible oversight vs D1/D2.")
print("  Checked D3's raw artist_name/track_name directly: 0 of 28,372 rows contain any")
print("  uppercase letter at all. The source data is already fully lowercase, so this was")
print("  never a real data-quality bug -- lowercase=True is now set for D3 anyway, purely")
print("  as defensive code in case a future raw CSV update isn't pre-lowercased.")

[STEP 3] Cleaning Health Report（清理前後對照）

  D1

=== D1 artist: 'artist' -> 'artist_clean' ===
  Changed: 328,951 rows (99.66%)


    'Adele'  ->  'adele'
    'The Kid LAROI & Justin Bieber'  ->  'the kid laroi & justin bieber'
    'Lil Nas X & Jack Harlow'  ->  'lil nas x & jack harlow'
    'Walker Hayes'  ->  'walker hayes'
    'Ed Sheeran'  ->  'ed sheeran'

=== D1 song: 'song' -> 'song_clean' ===


  Changed: 328,971 rows (99.66%)
    'Easy On Me'  ->  'easy on me'
    'Stay'  ->  'stay'
    'Industry Baby'  ->  'industry baby'
    'Fancy Like'  ->  'fancy like'
    'Bad Habits'  ->  'bad habits'

  D2

=== D2 artist: 'artist' -> 'artist_clean' ===
  Changed: 40,937 rows (99.59%)


    'Garland Green'  ->  'garland green'
    'Serge Gainsbourg'  ->  'serge gainsbourg'
    'Lord Melody'  ->  'lord melody'
    'Celia Cruz'  ->  'celia cruz'
    'P. Susheela'  ->  'p. susheela'

=== D2 song: 'song' -> 'song_clean' ===
  Changed: 40,899 rows (99.5%)
    'Jealous Kind Of Fella'  ->  'jealous kind of fella'
    'Initials B.B.'  ->  'initials b.b.'
    'Melody Twist'  ->  'melody twist'
    'Mi Bomba Sonó'  ->  'mi bomba sonó'
    'Uravu Solla'  ->  'uravu solla'

  D3

=== D3 artist: 'artist' -> 'artist_clean' ===
  Changed: 1 rows (0.0%)
    'babes in toyland '  ->  'babes in toyland'

=== D3 song: 'song' -> 'song_clean' ===
  Changed: 408 rows (1.44%)
    'don't look back (feat. van morrison)'  ->  'don't look back'
    'he don't love you [like i love you]'  ->  'he don't love you'
    'you're the song [that i can't stop singing]'  ->  'you're the song'
    'whenever i call you "friend" (feat. stevie nicks)'  ->  'whenever i call you "friend"'
    'i just can't stop 

---
## ③ Step 4 — 建立 Join Key（對應藍圖表 Step 4）

`join_key = artist_clean + "|" + song_clean`，用 `|` 當分隔符（歌手/歌名本身不太可能出現這個字元），04 合併資料時直接比對這個 key 就好，不用再各自比對 artist 跟 song 兩欄。

In [7]:
from IPython.display import display

print("[STEP 4] Join Key Creation（建立 Join Key）")

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    df["join_key"] = df["artist_clean"] + "|" + df["song_clean"]
    print(f"\n{dataset_name} join_key samples:")
    display(df[[cfg["artist_col"], cfg["song_col"], "artist_clean", "song_clean", "join_key"]]
            .drop_duplicates(subset="join_key")
            .head(5))

[STEP 4] Join Key Creation（建立 Join Key）

D1 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,Adele,Easy On Me,adele,easy on me,adele|easy on me
1,The Kid LAROI & Justin Bieber,Stay,the kid laroi & justin bieber,stay,the kid laroi & justin bieber|stay
2,Lil Nas X & Jack Harlow,Industry Baby,lil nas x & jack harlow,industry baby,lil nas x & jack harlow|industry baby
3,Walker Hayes,Fancy Like,walker hayes,fancy like,walker hayes|fancy like
4,Ed Sheeran,Bad Habits,ed sheeran,bad habits,ed sheeran|bad habits



D2 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,Garland Green,Jealous Kind Of Fella,garland green,jealous kind of fella,garland green|jealous kind of fella
1,Serge Gainsbourg,Initials B.B.,serge gainsbourg,initials b.b.,serge gainsbourg|initials b.b.
2,Lord Melody,Melody Twist,lord melody,melody twist,lord melody|melody twist
3,Celia Cruz,Mi Bomba Sonó,celia cruz,mi bomba sonó,celia cruz|mi bomba sonó
4,P. Susheela,Uravu Solla,p. susheela,uravu solla,p. susheela|uravu solla



D3 join_key samples:


,artist,song,artist_clean,song_clean,join_key
0,mukesh,mohabbat bhi jhoothi,mukesh,mohabbat bhi jhoothi,mukesh|mohabbat bhi jhoothi
4,frankie laine,i believe,frankie laine,i believe,frankie laine|i believe
6,johnnie ray,cry,johnnie ray,cry,johnnie ray|cry
10,pérez prado,patricia,pérez prado,patricia,pérez prado|patricia
12,giorgos papadopoulos,apopse eida oneiro,giorgos papadopoulos,apopse eida oneiro,giorgos papadopoulos|apopse eida oneiro


---
## ③ Step 5 — 跨資料集 Key 重疊率預測（對應藍圖表 Step 5）

04 合併之前先看一下：D1 的 join_key 有多少比例能在 D2/D3 裡找到對應。R 版本最後有一句重要備註：**D1-D3 的精確比對率大約只有 10.6%**，這裡重新算一次確認數字有沒有變。

In [8]:
print("[STEP 5] Cross-Dataset Key Overlap（跨資料集 Key 重疊率）")

d1_keys = set(DATASETS["D1"]["df"]["join_key"].unique())
d2_keys = set(DATASETS["D2"]["df"]["join_key"].unique())
d3_keys = set(DATASETS["D3"]["df"]["join_key"].unique())

d1_d2_overlap = len(d1_keys & d2_keys)
d1_d3_overlap = len(d1_keys & d3_keys)

print(f"  D1 unique keys: {len(d1_keys):,}")
print(f"  D2 unique keys: {len(d2_keys):,}")
print(f"  D3 unique keys: {len(d3_keys):,}\n")

print(f"  D1 ∩ D2 overlap: {d1_d2_overlap:,} keys ({round(d1_d2_overlap / len(d1_keys) * 100, 2)}% of D1)")
print(f"  D1 ∩ D3 overlap: {d1_d3_overlap:,} keys ({round(d1_d3_overlap / len(d1_keys) * 100, 2)}% of D1)")

[STEP 5] Cross-Dataset Key Overlap（跨資料集 Key 重疊率）
  D1 unique keys: 29,671
  D2 unique keys: 39,851
  D3 unique keys: 28,342

  D1 ∩ D2 overlap: 20,123 keys (67.82% of D1)
  D1 ∩ D3 overlap: 3,155 keys (10.63% of D1)


---
## ④ Step 6 — 存檔（對應藍圖表 Step 6）

存成 `.pkl`，讓 04 合併資料的階段能接著讀。跟 01 的 Step 8 一樣的邏輯：沒給路徑就跳過，不會意外覆蓋。

In [9]:
print("[STEP 6] Save Wrangled Data（存檔）")

for dataset_name, cfg in DATASETS.items():
    save_path = cfg.get("save_path")
    if save_path:
        cfg["df"].to_pickle(save_path)
        print(f"  [OK]  {dataset_name} saved to: {save_path}")
    else:
        print(f"  {dataset_name}: no save_path given, skipping.")

[STEP 6] Save Wrangled Data（存檔）


  [OK]  D1 saved to: ..\Data\wrangled_D1.pkl
  [OK]  D2 saved to: ..\Data\wrangled_D2.pkl
  [OK]  D3 saved to: ..\Data\wrangled_D3.pkl
